# Delta Lake on MinIO
Run the cells in order. The notebook uses the shared Spark worker, Hive Metastore, and `warehouse` bucket.

In [ ]:
print("start on Jupyter server")

start on Jupyter server


In [2]:
from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession

builder = (SparkSession.builder.appName('delta-notebook')
    .master('spark://spark-master:7077')
    .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog')
    .config('spark.hadoop.hive.metastore.uris', 'thrift://hive-metastore:9083')
    .config('spark.jars.ivy', '/tmp/.ivy2')
    .config('spark.hadoop.fs.s3a.endpoint', 'http://minio:9000')
    .config('spark.hadoop.fs.s3a.access.key', 'minioadmin')
    .config('spark.hadoop.fs.s3a.secret.key', 'minioadmin')
    .config('spark.hadoop.fs.s3a.aws.credentials.provider', 'org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider')
    .config('spark.hadoop.fs.s3a.endpoint.region', 'us-east-1')
    .config('spark.hadoop.fs.s3a.path.style.access', 'true')
    .config('spark.hadoop.fs.s3a.connection.ssl.enabled', 'false')
    .config('spark.hadoop.fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem'))
spark = configure_spark_with_delta_pip(
    builder,
    extra_packages=['org.apache.hadoop:hadoop-aws:3.3.4'],
).enableHiveSupport().getOrCreate()

## Write, append, and query

In [3]:
path = 's3a://warehouse/notebook/people'
df = spark.createDataFrame([(1, 'Ada', 'engineering'), (2, 'Grace', 'data')], ['id', 'name', 'team'])
df.write.format('delta').mode('overwrite').save(path)
df2 = spark.createDataFrame([(3, 'Lin', 'platform')], ['id', 'name', 'team'])
df2.write.format('delta').mode('append').save(path)
spark.read.format('delta').load(path).orderBy('id').show()

+---+-----+-----------+
| id| name|       team|
+---+-----+-----------+
|  1|  Ada|engineering|
|  2|Grace|       data|
|  3|  Lin|   platform|
+---+-----+-----------+



## Schema evolution

In [4]:
schema_path = 's3a://warehouse/notebook/people_schema'
spark.createDataFrame([(1, 'Ada')], ['id', 'name']).write.format('delta').mode('overwrite').save(schema_path)
spark.createDataFrame([(2, 'Grace', 'data')], ['id', 'name', 'team']).write.format('delta').option('mergeSchema', 'true').mode('append').save(schema_path)
spark.read.format('delta').load(schema_path).printSchema()

root
 |-- id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- team: string (nullable = true)



## Transaction history and time travel

In [5]:
spark.sql(f"DESCRIBE HISTORY '{path}'").select('version', 'operation').show()
spark.read.format('delta').option('versionAsOf', 0).load(path).show()

+-------+---------+
|version|operation|
+-------+---------+
|      1|    WRITE|
|      0|    WRITE|
+-------+---------+

+---+-----+-----------+
| id| name|       team|
+---+-----+-----------+
|  1|  Ada|engineering|
|  2|Grace|       data|
+---+-----+-----------+



In [6]:
spark.stop()